# Chapter 4: Representing Data and Feature Engineering

## Overview
This notebook covers methods for preparing, engineering, and selecting features for machine learning models:
1. **Categorical Variable Encoding**: One-Hot Encoding and Column Transformers.
2. **Binning & Discretization**: Splitting continuous features to increase non-linear capacity in linear models.
3. **Interactions and Polynomials**: Generating feature combinations with `PolynomialFeatures`.
4. **Univariate Non-Linear Transformations**: Adjusting skewed distributions using mathematical transformations.
5. **Automatic Feature Selection**: Univariate statistics, model-based selection, and Recursive Feature Elimination (RFE).

## 1. Encoding Categorical Data

Machine learning estimators require numerical inputs. Categorical variables must be converted into numerical representations:
- **One-Hot Encoding ($1$-of-$K$)**: Replaces a categorical feature with $K$ binary indicator features where exactly one bit is active ($1$) per sample.
- **`ColumnTransformer`**: Applies targeted transformations (e.g., scaling to numeric, encoding to categoricals) across specific DataFrame column subsets.

In [ ]:
# Construct sample dataframe containing mixed types
df_housing = pd.DataFrame({
    "Age": [25, 45, 35, 50, 23],
    "Income": [50000, 85000, 62000, 110000, 48000],
    "Neighborhood": ["Downtown", "Suburbs", "Suburbs", "Rural", "Downtown"],
    "PropertyType": ["Condo", "House", "Condo", "House", "Apartment"]
})

print("Original Data:")
display(df_housing)

# Define column groups
num_features = ["Age", "Income"]
cat_features = ["Neighborhood", "PropertyType"]

# Build preprocessor pipeline via ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(sparse_output=False, drop="first"), cat_features)
    ]
)

X_processed = preprocessor.fit_transform(df_housing)
feature_names = num_features + list(preprocessor.named_transformers_["cat"].get_feature_names_out(cat_features))

df_transformed = pd.DataFrame(X_processed, columns=feature_names)
print("\nTransformed Feature Matrix (Scaled Numeric + Drop-First One-Hot Encoded Categories):")
display(df_transformed)

## 2. Binning, Discretization, and Linear Models

Linear models fit a single global weight per feature, restricting them to linear boundaries.
**Discretization (Binning)** splits a continuous feature into discrete intervals, allowing linear models to fit distinct step constants across different feature ranges:

$$\hat{y} = \sum_{j=1}^{B} w_j \mathbb{I}(x \in \text{Bin}_j)$$

In [ ]:
# Generate synthetic non-linear 1D continuous dataset
np.random.seed(42)
X_single = np.linspace(-3, 3, 100).reshape(-1, 1)
y_single = np.sin(X_single).ravel() + np.random.normal(0, 0.1, size=100)

# 1. Fit standard linear regression
line_grid = np.linspace(-3, 3, 1000).reshape(-1, 1)
lr_raw = LinearRegression().fit(X_single, y_single)

# 2. Fit linear regression on binned features
kb = KBinsDiscretizer(n_bins=10, strategy="uniform", encode="onehot")
X_binned = kb.fit_transform(X_single)
line_binned = kb.transform(line_grid)

lr_binned = LinearRegression().fit(X_binned, y_single)

# Visualization comparison
plt.figure(figsize=(10, 4))
plt.plot(X_single, y_single, "o", color="navy", alpha=0.5, label="Data")
plt.plot(line_grid, lr_raw.predict(line_grid), "--", label="Linear Regression (Raw Input)", color="crimson")
plt.plot(line_grid, lr_binned.predict(line_binned), label="Linear Regression (10 Bins)", color="teal", lw=2)
plt.xlabel("Input Feature")
plt.ylabel("Target Output")
plt.title("Impact of Feature Discretization on Linear Models")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Interaction Features & Polynomial Expansion

While binning splits features into isolated ranges, **Polynomial Features** adds product interaction terms across continuous dimensions:

$$\phi(x_1, x_2) = [1, x_1, x_2, x_1^2, x_1 x_2, x_2^2]$$

Adding interactions enables linear models to fit smooth curves and multiplicative relationships.

In [ ]:
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X_single)
line_poly = poly.transform(line_grid)

lr_poly = LinearRegression().fit(X_poly, y_single)

plt.figure(figsize=(10, 4))
plt.plot(X_single, y_single, "o", color="navy", alpha=0.5, label="Data")
plt.plot(
    line_grid,
    lr_poly.predict(line_poly),
    color="darkorange",
    lw=2,
    label="Linear Reg + Polynomial Degree 3",
)
plt.xlabel("Input Feature")
plt.ylabel("Target Output")
plt.title("Polynomial Feature Expansion for Smooth Non-Linear Approximations")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Univariate Non-Linear Transformations

Linear algorithms perform best when input feature distributions are roughly Gaussian (bell-shaped).
For heavily right-skewed data (e.g., counting data, incomes, frequency metrics), applying logarithmic functions compresses long right tails:

$$x_{\text{transformed}} = \log(x + 1)$$

In [ ]:
# Generate Poisson-distributed right-skewed data
np.random.seed(42)
X_skewed = np.random.poisson(lam=2, size=(1000, 1))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# Plot raw distribution
axes[0].hist(X_skewed, bins=15, color="crimson", edgecolor="black", alpha=0.7)
axes[0].set_title("Original Skewed Feature Distribution")
axes[0].set_xlabel("Value Range")
axes[0].set_ylabel("Frequency")

# Plot log-transformed distribution
X_log = np.log1p(X_skewed)
axes[1].hist(X_log, bins=15, color="teal", edgecolor="black", alpha=0.7)
axes[1].set_title("Log-Transformed Distribution: log(x + 1)")
axes[1].set_xlabel("Transformed Value Range")

plt.tight_layout()
plt.show()

## 5. Automatic Feature Selection

High-dimensional feature spaces increase model variance and risk overfitting. We evaluate three feature selection strategies:
1. **Univariate Statistics**: Computes statistically significant relationship scores (ANOVA $F$-test or $\chi^2$) per individual feature independently.
2. **Model-Based Selection**: Uses an estimator with feature importance or coefficient weights (e.g., Lasso, Random Forest) to drop sub-threshold features.
3. **Recursive Feature Elimination (RFE)**: Iteratively trains models, pruning the least important feature in each round until reaching a target feature count.

In [ ]:
cancer = load_breast_cancer()

# Introduce 50 noisy random features to test selector effectiveness
np.random.seed(42)
noise = np.random.normal(size=(len(cancer.data), 50))
X_noisy = np.hstack([cancer.data, noise])

X_tr, X_te, y_tr, y_te = train_test_split(
    X_noisy, cancer.target, test_size=0.3, random_state=42
)

print(f"Dataset Shape with Noise Included: {X_noisy.shape}")

# 1. Univariate Selection (Select Top 30% via ANOVA F-test)
select_univariate = SelectPercentile(score_func=f_classif, percentile=30)
X_tr_uni = select_univariate.fit_transform(X_tr, y_tr)
X_te_uni = select_univariate.transform(X_te)

logreg_uni = LogisticRegression(max_iter=10000).fit(X_tr_uni, y_tr)
print("\n1. Univariate Selection (30% Features Retained):")
print(f"   Selected Features Count: {X_tr_uni.shape[1]}")
print(f"   Test Set Accuracy:       {logreg_uni.score(X_te_uni, y_te)*100:.2f}%")

# 2. Model-Based Selection (Random Forest Thresholding)
select_model = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=42), threshold="median"
)
X_tr_mb = select_model.fit_transform(X_tr, y_tr)
X_te_mb = select_model.transform(X_te)

logreg_mb = LogisticRegression(max_iter=10000).fit(X_tr_mb, y_tr)
print("\n2. Model-Based Selection (Median Importance Cutoff):")
print(f"   Selected Features Count: {X_tr_mb.shape[1]}")
print(f"   Test Set Accuracy:       {logreg_mb.score(X_te_mb, y_te)*100:.2f}%")

# 3. Recursive Feature Elimination (RFE with Random Forest)
select_rfe = RFE(
    RandomForestClassifier(n_estimators=100, random_state=42),
    n_features_to_select=30,
)
X_tr_rfe = select_rfe.fit_transform(X_tr, y_tr)
X_te_rfe = select_rfe.transform(X_te)

logreg_rfe = LogisticRegression(max_iter=10000).fit(X_tr_rfe, y_tr)
print("\n3. Recursive Feature Elimination (RFE to 30 Features):")
print(f"   Selected Features Count: {X_tr_rfe.shape[1]}")
print(f"   Test Set Accuracy:       {logreg_rfe.score(X_te_rfe, y_te)*100:.2f}%")